# Nạp Online Retail II vào SQLite một lần, truy vấn mãi mãi

**Bối cảnh.** Team phân tích trả lời câu hỏi về cửa hàng quà tặng online (Online Retail II, UCI, CC BY 4.0) bằng cách
mở file Excel 2 sheet, hơn một triệu dòng. Mỗi lần `pd.read_excel` mất khoảng 40 giây một sheet, ba người ba notebook
ra ba con số doanh thu khác nhau (người ghép hai sheet quên bỏ đoạn trùng, người quên bỏ hoá đơn huỷ). Bạn đề xuất:
nạp một lần vào một database SQL nhỏ, có nhật ký nạp, rồi mọi người truy vấn cùng một bảng bằng cùng một câu SQL.

Bạn dùng **SQLAlchemy** với **SQLite** (một tệp, không cần server). Cùng code chạy được trên Postgres chỉ bằng cách đổi
URL kết nối: test cuối cùng kiểm điều đó khi dịch vụ `postgres` của repo đang chạy.

Yêu cầu của một lần nạp tốt:

- **Idempotent theo batch**: nạp lại batch `2010-2011` không nhân đôi dòng (xoá batch cũ rồi ghi, trong cùng một
  transaction);
- **Atomic**: lỗi giữa chừng thì database giữ nguyên trạng thái trước đó;
- **Có nhật ký**: bảng `load_log` ghi batch nào, từ nguồn nào, bao nhiêu dòng, lúc nào;
- **Đối chiếu được**: số dòng và doanh thu trong database khớp với nguồn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc tệp Excel thật (khoảng 1–2 phút), nạp vào `data/clean/online_retail.sqlite` và truy vấn.
- Chấm: `uv run pytest tracks/data-analyst/modules/05-data-collection/exercises/intermediate/02-retail-sqlite-load -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

XLSX = ensure("online-retail-ii") / "online_retail_II.xlsx"
DB = ROOT / "data/clean/online_retail.sqlite"
DB.parent.mkdir(parents=True, exist_ok=True)
print(XLSX, f"{XLSX.stat().st_size / 1e6:.1f} MB")

In [ ]:
import time
from datetime import datetime, timezone

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

## 1. Schema (cho sẵn)

Một bảng `sales` giữ mọi dòng hoá đơn kèm `batch_id` (batch nạp), một bảng `load_log` một dòng mỗi batch. Kiểu dữ liệu
viết sao cho cùng câu lệnh chạy được trên SQLite và Postgres. `create_schema(engine)` chạy từng câu lệnh trong một
transaction; chạy lại không sao (`IF NOT EXISTS`).

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS sales (
    batch_id     TEXT NOT NULL,
    invoice      TEXT NOT NULL,
    stock_code   TEXT,
    description  TEXT,
    quantity     INTEGER,
    invoice_date TIMESTAMP,
    price        DOUBLE PRECISION,
    customer_id  BIGINT,
    country      TEXT
);
CREATE INDEX IF NOT EXISTS ix_sales_batch ON sales (batch_id);
CREATE INDEX IF NOT EXISTS ix_sales_date ON sales (invoice_date);
CREATE TABLE IF NOT EXISTS load_log (
    batch_id  TEXT PRIMARY KEY,
    source    TEXT,
    n_rows    INTEGER,
    loaded_at TEXT
)
"""


def create_schema(engine: sa.Engine) -> None:
    with engine.begin() as conn:
        for statement in SCHEMA.split(";"):
            if statement.strip():
                conn.execute(text(statement))

## 2. Nạp một batch

`to_table(df) -> pd.DataFrame`: đổi tên 8 cột của bộ gốc sang tên trong bảng theo `COLUMNS` (đúng thứ tự đó), `invoice`
và `stock_code` thành chuỗi (`536365` và `C536379` cùng một kiểu), `quantity` số nguyên, `price` số thực,
`invoice_date` datetime, `customer_id` kiểu `Int64` (số nguyên cho phép thiếu). Các cột khác giữ nguyên.

`load_batch(engine, df, batch_id, source, now=...) -> int`: trong **một** transaction (`with engine.begin() as conn:`):

1. xoá mọi dòng `sales` có `batch_id` này;
2. ghi `to_table(df)` kèm cột `batch_id` vào `sales` (`DataFrame.to_sql(..., con=conn, if_exists="append",
   index=False, chunksize=50_000)`);
3. xoá rồi ghi lại dòng `load_log` của batch: `batch_id`, `source`, `n_rows`, `loaded_at = now()`.

Trả số dòng đã ghi. Mọi giá trị đưa vào SQL đi qua **bind parameter** (`text("... WHERE batch_id = :b")`, `{"b": ...}`),
không ghép chuỗi.

In [ ]:
COLUMNS = {"Invoice": "invoice", "StockCode": "stock_code", "Description": "description", "Quantity": "quantity",
           "InvoiceDate": "invoice_date", "Price": "price", "Customer ID": "customer_id", "Country": "country"}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def to_table(df: pd.DataFrame) -> pd.DataFrame:
    """8 cột tên snake_case, kiểu dữ liệu ổn định cho SQL."""
    out = df.rename(columns=COLUMNS)[list(COLUMNS.values())].copy()
    out["invoice"] = out["invoice"].astype(str)
    out["stock_code"] = out["stock_code"].astype(str)
    out["quantity"] = out["quantity"].astype("int64")
    out["price"] = out["price"].astype("float64")
    out["invoice_date"] = pd.to_datetime(out["invoice_date"])
    out["customer_id"] = out["customer_id"].astype("Int64")
    return out


def load_batch(engine: sa.Engine, df: pd.DataFrame, batch_id: str, source: str, now=utc_now) -> int:
    """Xoá batch cũ, ghi batch mới và load_log, tất cả trong một transaction."""
    rows = to_table(df)
    rows.insert(0, "batch_id", batch_id)
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM sales WHERE batch_id = :b"), {"b": batch_id})
        rows.to_sql("sales", con=conn, if_exists="append", index=False, chunksize=50_000)
        conn.execute(text("DELETE FROM load_log WHERE batch_id = :b"), {"b": batch_id})
        conn.execute(
            text("INSERT INTO load_log (batch_id, source, n_rows, loaded_at) VALUES (:b, :s, :n, :t)"),
            {"b": batch_id, "s": source, "n": len(rows), "t": now()},
        )
    return len(rows)

In [ ]:
# explore: đọc hai sheet (chậm - đây là lý do nạp một lần), bỏ đoạn trùng 01-09/12/2010
started = time.monotonic()
dtypes = {"Invoice": str, "StockCode": str, "Description": str}
sheet_0910 = pd.read_excel(XLSX, sheet_name="Year 2009-2010", dtype=dtypes)
sheet_1011 = pd.read_excel(XLSX, sheet_name="Year 2010-2011", dtype=dtypes)
read_seconds = time.monotonic() - started
print(f"đọc 2 sheet: {read_seconds:.0f} giây; {len(sheet_0910):,} + {len(sheet_1011):,} dòng")

overlap = sheet_0910[sheet_0910.InvoiceDate >= "2010-12-01"]
print(f"đoạn trùng trong sheet 2009-2010: {len(overlap):,} dòng, "
      f"{overlap.InvoiceDate.min()} -> {overlap.InvoiceDate.max()}")
sheet_0910 = sheet_0910[sheet_0910.InvoiceDate < "2010-12-01"]

In [ ]:
# explore: nạp hai batch, rồi nạp lại một batch để thấy idempotent
engine = sa.create_engine(f"sqlite:///{DB}")
create_schema(engine)
for batch, df in [("2009-2010", sheet_0910), ("2010-2011", sheet_1011)]:
    started = time.monotonic()
    n = load_batch(engine, df, batch, f"online_retail_II.xlsx#Year {batch}")
    print(f"batch {batch}: {n:,} dòng, {time.monotonic() - started:.1f} giây")
load_batch(engine, sheet_1011, "2010-2011", "online_retail_II.xlsx#Year 2010-2011")
with engine.connect() as conn:
    print("tổng dòng sau khi nạp lại:", conn.execute(text("SELECT COUNT(*) FROM sales")).scalar())
    print(pd.read_sql(text("SELECT * FROM load_log ORDER BY batch_id"), conn))
print(f"kích thước tệp SQLite: {DB.stat().st_size / 1e6:.0f} MB")

In [ ]:
# explore: vì sao không dùng method="multi" với SQLite - thử trên 50 000 dòng
probe = sa.create_engine("sqlite://")
create_schema(probe)
sample = to_table(sheet_1011.head(50_000)).assign(batch_id="probe")
for method in [None, "multi"]:
    started = time.monotonic()
    with probe.begin() as conn:
        sample.to_sql("sales", con=conn, if_exists="append", index=False, chunksize=1_000, method=method)
    print(f"method={method}: {time.monotonic() - started:.2f} giây")

## 3. Truy vấn và đối chiếu

`monthly_revenue(engine, start, end) -> pd.DataFrame`: doanh thu theo tháng trên dòng hợp lệ (định nghĩa chung: bỏ
`invoice` bắt đầu bằng `C`, bỏ `quantity <= 0` hoặc `price <= 0`), chỉ các dòng `start <= invoice_date < end` (`start`,
`end` là chuỗi ngày `YYYY-MM-DD`, truyền bằng bind parameter). Ba cột: `month` (chuỗi `YYYY-MM`), `revenue` (tổng
`quantity * price`), `invoices` (số hoá đơn hợp lệ khác nhau); sắp theo tháng. Viết SQL chạy được trên cả SQLite và
Postgres: lấy tháng bằng `substr(CAST(invoice_date AS TEXT), 1, 7)` (SQLite không có `date_trunc`, Postgres không có
`strftime`).

`reconcile(engine, df, batch_id) -> dict`: so nguồn `df` với batch trong database. Khoá: `rows_source`, `rows_db` (số
dòng), `revenue_source`, `revenue_db` (doanh thu dòng hợp lệ), `ok` (`True` khi số dòng bằng nhau và doanh thu lệch
không quá 0.01).

In [ ]:
VALID = "invoice NOT LIKE 'C%' AND quantity > 0 AND price > 0"

MONTHLY_SQL = f"""
SELECT substr(CAST(invoice_date AS TEXT), 1, 7) AS month,
       SUM(quantity * price)                     AS revenue,
       COUNT(DISTINCT invoice)                   AS invoices
FROM sales
WHERE {VALID}
  AND invoice_date >= :start AND invoice_date < :end
GROUP BY 1
ORDER BY 1
"""


def monthly_revenue(engine: sa.Engine, start: str, end: str) -> pd.DataFrame:
    """month, revenue, invoices trên dòng hợp lệ trong [start, end)."""
    with engine.connect() as conn:
        return pd.read_sql(text(MONTHLY_SQL), conn, params={"start": start, "end": end})


def reconcile(engine: sa.Engine, df: pd.DataFrame, batch_id: str) -> dict:
    """So số dòng và doanh thu hợp lệ giữa nguồn và batch trong database."""
    src = to_table(df)
    valid = src[~src["invoice"].str.startswith("C") & (src["quantity"] > 0) & (src["price"] > 0)]
    revenue_source = float((valid["quantity"] * valid["price"]).sum())
    with engine.connect() as conn:
        rows_db = conn.execute(text("SELECT COUNT(*) FROM sales WHERE batch_id = :b"), {"b": batch_id}).scalar()
        revenue_db = conn.execute(
            text(f"SELECT COALESCE(SUM(quantity * price), 0) FROM sales WHERE batch_id = :b AND {VALID}"),
            {"b": batch_id},
        ).scalar()
    revenue_db = float(revenue_db)
    return {
        "rows_source": len(src),
        "rows_db": int(rows_db),
        "revenue_source": revenue_source,
        "revenue_db": revenue_db,
        "ok": len(src) == rows_db and abs(revenue_source - revenue_db) <= 0.01,
    }

In [ ]:
# explore: đối chiếu từng batch, rồi doanh thu theo tháng (truy vấn mất bao lâu so với đọc Excel?)
for batch, df in [("2009-2010", sheet_0910), ("2010-2011", sheet_1011)]:
    print(batch, reconcile(engine, df, batch))
started = time.monotonic()
monthly = monthly_revenue(engine, "2009-12-01", "2012-01-01")
print(f"truy vấn: {time.monotonic() - started:.2f} giây (đọc Excel: {read_seconds:.0f} giây)")
print(monthly.round(2).to_string(index=False))

In [ ]:
# explore: cùng tháng hai năm, và tháng 12/2011 chưa đủ
import matplotlib.pyplot as plt

m = monthly.set_index("month")
print(f"11/2011 so với 11/2010: {m.loc['2011-11', 'revenue'] / m.loc['2010-11', 'revenue'] - 1:+.1%}")
fig, ax = plt.subplots(figsize=(8, 4))
full = m.drop(index="2011-12")
ax.plot(full.index, full.revenue / 1e3, marker="o", color="#4c78a8")
ax.plot(["2011-11", "2011-12"], m.loc[["2011-11", "2011-12"], "revenue"] / 1e3, ls=":", color="#999")
ax.annotate("12/2011: chỉ tới ngày 9", ("2011-12", m.loc["2011-12", "revenue"] / 1e3), xytext=(-150, -10),
            textcoords="offset points")
ax.set_ylabel("doanh thu (nghìn GBP)")
ax.set_xticks(range(0, len(m), 3))
ax.set_title("Doanh thu theo tháng, 12/2009 - 12/2011 (từ SQLite)", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 4. Đọc kết quả, rồi làm trắc nghiệm

1. Ghép hai sheet mà không bỏ đoạn trùng thì thừa bao nhiêu dòng? Doanh thu tháng 12/2010 sai thế nào?
2. Nạp lại batch `2010-2011` thì tổng dòng có đổi không? Nhờ đâu?
3. Đọc Excel mất bao lâu, truy vấn SQLite mất bao lâu? Khi nào SQLite không còn đủ (nhiều người ghi cùng lúc, dữ liệu
   hàng chục GB) và nên chuyển sang Postgres/warehouse?
4. Tháng 11/2011 so với tháng 11/2010 thế nào? Vì sao không so tháng 12/2011 với tháng 12/2010?

Xong thì mở app, làm phần **trắc nghiệm** của bài.